# Master — KIČMENA MOŽDINA (nnU-Net)

Model za lezije u kičmenoj moždini na MS-Multi-Spine skupu.

## Ćelija 0 — priprema

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q nnunetv2 nibabel

import os, shutil, json, glob, re
import numpy as np
import nibabel as nib

MASTER = '/content/drive/MyDrive/Master/kicma'
os.environ['nnUNet_raw']          = '/content/nnUNet_raw'          
os.environ['nnUNet_preprocessed'] = '/content/nnUNet_preprocessed' 
os.environ['nnUNet_results']      = f'{MASTER}/modeli/nnUNet_results'
for p in [os.environ['nnUNet_raw'], os.environ['nnUNet_preprocessed'], os.environ['nnUNet_results']]:
    os.makedirs(p, exist_ok=True)

import torch
print('GPU:', torch.cuda.is_available(), '(za pripremu nije nužan)')
print('Spremno.')

Mounted at /content/drive
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
GPU: True (za pripremu nije nužan)
Spremno.


In [ ]:
import os, torch, nnunetv2
vdir = os.path.join(os.path.dirname(nnunetv2.__file__),
                    'training','nnUNetTrainer','variants','training_length')
with open(os.path.join(vdir, 'nnUNetTrainer_100epochs_save10.py'), 'w') as f:
    f.write('''import torch
from nnunetv2.training.nnUNetTrainer.variants.training_length.nnUNetTrainer_Xepochs import nnUNetTrainer_100epochs

class nnUNetTrainer_100epochs_save10(nnUNetTrainer_100epochs):
    def __init__(self, plans, configuration, fold, dataset_json, device=torch.device('cuda')):
        super().__init__(plans, configuration, fold, dataset_json, device)
        self.save_every = 10
''')
print('trener popravljen — sad ponovo pokreni Ćeliju 4')

trener popravljen — sad ponovo pokreni Ćeliju 4


## Ćelija 1 — raspakivanje + pregled strukture


In [ ]:
import tarfile
tar_path = f'{MASTER}/podaci_sirovi/MS_MultiSpine_dataset.tar.gz'
print('tar postoji:', os.path.exists(tar_path),
      '| GB:', round(os.path.getsize(tar_path)/1e9,2) if os.path.exists(tar_path) else '-')

if not os.path.exists('/content/multispine'):
    print('raspakivam (nekoliko min)...')
    with tarfile.open(tar_path) as t:
        t.extractall('/content/multispine')
    print('gotovo.')

base = '/content/multispine'

print('\n--- STABLO ---')
for root, dirs, files in sorted(os.walk(base)):
    dirs[:] = [d for d in dirs if not d.startswith('__')]
    rel = os.path.relpath(root, base)
    depth = 0 if rel=='.' else rel.count(os.sep)+1
    if depth > 3:
        dirs[:] = []; continue
    ime = 'ROOT' if rel=='.' else os.path.basename(root)
    print('   '*depth + ime + '/   ' + (str(sorted(files)[:8]) if files else ''))

# klasifikacija fajlova
svi = glob.glob(f'{base}/**/*.nii*', recursive=True)
def je(f, *k): n=os.path.basename(f).lower(); return any(x in n for x in k)
maske = [f for f in svi if je(f,'lesion','_les','mask','seg','_gt','label')]
print('\n--- KLASIFIKACIJA ---')
print('ukupno .nii:', len(svi), '| kandidata maski:', len(maske))
for seq in ['t2','stir','psir','mp2rage','t1']:
    n = [f for f in svi if je(f,seq) and f not in maske]
    print(f'  {seq.upper()}: {len(n)}   primer: {os.path.basename(n[0]) if n else "-"}')
print('  primer maske:', os.path.basename(maske[0]) if maske else '-')

tar postoji: True | GB: 5.71
raspakivam (nekoliko min)...


/tmp/ipykernel_1288/2539177159.py:9: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  t.extractall('/content/multispine')


gotovo.

--- STABLO ---
ROOT/   
   MS_MultiSpine_dataset/   ['DUA_MRI_Data_Access_Agreement_MICCAI_2025.pdf', 'README.md', 'case_characteristics.tsv']
      derivatives/   
         preprocessed/   
         preprocessedAndRegistered/   
      rawdata/   
         sub-001/   ['11-001_LESIONMASK.nii.gz', '11-001_STIR.nii.gz', '11-001_T2.nii.gz']
         sub-002/   ['11-002_LESIONMASK.nii.gz', '11-002_STIR.nii.gz', '11-002_T2.nii.gz']
         sub-003/   ['11-003_LESIONMASK.nii.gz', '11-003_STIR.nii.gz', '11-003_T2.nii.gz']
         sub-004/   ['11-004_LESIONMASK.nii.gz', '11-004_STIR.nii.gz', '11-004_T2.nii.gz']
         sub-005/   ['11-005_LESIONMASK.nii.gz', '11-005_STIR.nii.gz', '11-005_T2.nii.gz']
         sub-006/   ['11-006_LESIONMASK.nii.gz', '11-006_STIR.nii.gz', '11-006_T2.nii.gz']
         sub-007/   ['11-007_LESIONMASK.nii.gz', '11-007_STIR.nii.gz', '11-007_T2.nii.gz']
         sub-008/   ['11-008_LESIONMASK.nii.gz', '11-008_STIR.nii.gz', '11-008_T2.nii.gz']
         sub-00

## Ćelija 2 — pravljenje nnU-Net formata (Dataset011_Spine)
V1: ulaz = **T2**, izlaz = **lezijska maska**. Sam pronalazi parove (T2 + maska istog pacijenta) i ispiše koliko ih je. 

In [ ]:
import glob, os, shutil, json
import numpy as np, nibabel as nib

shutil.rmtree(os.environ['nnUNet_raw'] + '/Dataset011_Spine', ignore_errors=True)  

base = '/content/multispine/MS_MultiSpine_dataset'
kandidati = [f'{base}/derivatives/preprocessedAndRegistered',
             f'{base}/derivatives/preprocessed',
             f'{base}/rawdata']
IZVOR = next((c for c in kandidati if os.path.isdir(c) and glob.glob(f'{c}/sub-*/*.nii*')), None)
print('koristim izvor:', IZVOR)
assert IZVOR, 'Nijedan izvor nema podatke!'

def nadji(folder, *k, ne=()):
    for f in sorted(os.listdir(folder)):
        n = f.lower()
        if n.endswith(('.nii','.nii.gz')) and any(x in n for x in k) and not any(y in n for y in ne):
            return os.path.join(folder, f)
    return None

parovi = []
for sub in sorted(os.listdir(IZVOR)):
    sdir = os.path.join(IZVOR, sub)
    if not os.path.isdir(sdir) or not sub.startswith('sub-'): continue
    t2   = nadji(sdir, 't2', ne=('lesion','mask','seg'))
    mask = nadji(sdir, 'lesionmask', 'mask', 'lesion', 'seg')
    if t2 and mask:
        parovi.append((sub, t2, mask))
print('pacijenata (T2+maska):', len(parovi))
if parovi:
    print('primer:', parovi[0][1].replace(base+'/',''), '|', parovi[0][2].replace(base+'/',''))
assert parovi, 'Nema parova.'

DS = os.environ['nnUNet_raw'] + '/Dataset011_Spine'
for s in ['imagesTr','labelsTr']: os.makedirs(f'{DS}/{s}', exist_ok=True)

def upisi_masku(src, dst):
    img = nib.load(src)
    d = (np.asarray(img.dataobj) > 0).astype(np.uint8)
    nib.save(nib.Nifti1Image(d, img.affine, img.header), dst)

n = 0
for sub, t2, mask in parovi:
    cid = sub.replace('-','_')
    shutil.copy(t2, f'{DS}/imagesTr/{cid}_0000.nii.gz')
    upisi_masku(mask, f'{DS}/labelsTr/{cid}.nii.gz')
    n += 1

with open(f'{DS}/dataset.json','w') as f:
    json.dump({"channel_names":{"0":"T2"},"labels":{"background":0,"lesion":1},
               "numTraining":n,"file_ending":".nii.gz",
               "overwrite_image_reader_writer":"SimpleITKIO"}, f, indent=2)
print('upisano:', n, '| dataset.json OK')

koristim izvor: /content/multispine/MS_MultiSpine_dataset/derivatives/preprocessedAndRegistered
pacijenata (T2+maska): 100
primer: derivatives/preprocessedAndRegistered/sub-001/11-001_T2.nii.gz | derivatives/preprocessedAndRegistered/sub-001/11-001_LESIONMASK.nii.gz
upisano: 100 | dataset.json OK


## Ćelija 3 — nnU-Net plan & preprocess

In [ ]:
!nnUNetv2_plan_and_preprocess -d 11 --verify_dataset_integrity

Fingerprint extraction...
Dataset011_Spine
Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> reader/writer

####################
verify_dataset_integrity Done. 
If you didn't see any error messages then your dataset is most likely OK!
####################

Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> reader/writer
Extracting dataset fingerprint: 100% 100/100 [04:34<00:00,  2.75s/it]
Experiment planning...

############################
INFO: You are using the old nnU-Net default planner. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Attempting to find 3d_lowres config. 
Current spacing: [0.515      0.51499997 0.515     ]. 
Current patch size: (np.int64(256), np.int64(64), np.int64(128)). 
Current median shape: [582.03883495  80.58252427 185.9223301 ]
Attempting to find 3d_lowres config. 
Curren

## Ćelija 4 — TRENING (GPU)

In [ ]:
rez = f"{os.environ['nnUNet_results']}/Dataset011_Spine/nnUNetTrainer_100epochs_save10__nnUNetPlans__3d_fullres/fold_0"
nastavi = "--c" if os.path.exists(f"{rez}/checkpoint_latest.pth") else ""
print("Nastavljam od checkpointa." if nastavi else "Krećem od nule.")
!nnUNetv2_train 11 3d_fullres 0 -tr nnUNetTrainer_100epochs_save10 {nastavi}

Nastavljam od checkpointa.

############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

2026-10-28 08:06:16.149542: Using torch.compile...
2026-10-28 08:06:27.794578: do_dummy_2d_data_aug: False
2026-10-28 08:06:27.798940: Creating new 5-fold cross-validation split...
2026-10-28 08:06:27.805241: Desired fold for training: 0
2026-10-28 08:06:27.808193: Thi